This tutorial guides you through the design of custom probes against a gene of interest.

First, we need to specify the gene symbol, and which feature we are interested in designing probes agains (in this case we are using exons).
We also need to specify an output directory for the analysis.

### 1. Specify parameters

In [1]:
import os
import warnings
import subprocess
import pandas as pd
import numpy as np
import pybedtools
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction
from gene2probe import *


We can now define the genes to process and the common output root. Each gene will be written into its own subdirectory under this root, so the notebook can be run once for a whole list of Gene IDs.

In [ ]:
## Specify genes of interest and feature of interest
gene_ids = ["CDKN2B-AS1", "XIST", "GAS5", "LNCTAM34A", "H19", "MIR4435-2HG"]
mode = 'transcript' ## Whether to consider only exons / introns or full gene
output_root = '../sample_run'
os.makedirs(output_root, exist_ok=True)

Additionally, we need to provide the path to several resource files. Many of these files can be obtained from [UCSC table browser](https://genome.ucsc.edu/cgi-bin/hgTables).

We also need a blast database, such as the one we generated in the [previous tutorial](https://github.com/Teichlab/gene2probe/blob/main/notebooks/001_make_blast_database.ipynb).

In [ ]:
## Required resources (most can be downloaded from 
gtf = '../hg38_resources/hg38.ncbiRefSeq.gtf' ## Gene annotation in gtf file
## We recommend using RefSeq as this is manually curated and more likely to contain an isoform that is present across most cell types
## Alternatively, one can filter based on RNA-seq data for a cell type/tissue of interest
fasta = '../hg38_resources/hg38.fa' ## Genome in fasta file
snp_db = '../hg38_resources/hg38_snp151Common.bed' ## Database of known SNPs and small indels
repeats = '../hg38_resources/hg38_rmsk.bed' ## bed file with repeats/low complexity regions to be excluded
gaps = '../hg38_resources/hg38_rmsk.bed' ## bed file with gaps in the genome assembly to be excluded
blast_db = '../hg38_resources/001_blastdb/hg38_ncbiRefSeq_transcripts_db' ## Database of all human transcripts to blast against

gene_anno = read_gtf(gtf)

In [4]:
## Path to blast binaries.
## Replace with your conda environment
## This can also be omitted if you started the jupyter session from within the gene2probe conda environment
print('current working directory:', os.getcwd())

blast_exec_path = f"{os.environ['HOME']}/.miniforge3/envs/gene2probe_env/bin/"

if not os.path.isdir(blast_exec_path):
    warnings.warn(
        f'BLAST executable directory not found: {blast_exec_path}',
        RuntimeWarning,
    )

print('blast executable path:', blast_exec_path)


current working directory: /Users/dangriffiths/gene2probe/notebooks
blast executable path: /Users/dangriffiths/.miniforge3/envs/gene2probe_env/bin/


Finally, we need to provide a set of parameters related to our probe's length, at which nucleotide it's split (if at all), the acceptable range for GC content and any specific requirements for individual nucleotides.

Here we are following the [recommendations of 10x Genomics for custom probes for VisiumHD/VisiumFFPE/Flex](https://cdn.10xgenomics.com/image/upload/v1697739385/support-documents/CG000621_CustomProbeDesign_TechNote_RevC.pdf).

In [ ]:
## Additional parameters regarding how the probe should look like
probe_length = 50 ## Length of probe in nucleotides
split_nt = 25 ## Index of nucleotide to split the probe at (start of RHS) - set to None if splitting probe is not needed
min_GC = 0.44 ## Minimum GC content for probe (if split probe, applied to both LHS and RHS)
max_GC = 0.72 ## Maximum GC content for probe (if split probe, applied to both LHS and RHS)
required_nts = {24: 'T'} ## Dictionary of index (0-based) for required nts - by default, 25th nucleotide must be a T - set to None if no requirements
probe_offset = 100 ## Minimum distance between probes - 10 bp is the recommended minimum by 10x, this can also be adjusted depending on how many probes pass other cutoffs
probe_selection_offset = 1000 ## Minimum distance between the selected probes
n_desired_probes = 3 ## Number of probes to be designed.
min_mismatches = 5 ## Minimum number of mismatches (in at least LHS or RHS) - here we require in both to be more conservative


In [6]:
## Optionally, we can also specify adapters that have to be added to the probes.
## For example, for visiumHD:
LHS_pref = 'CCTTGGCACCCGAGAATTCCA' ## Will be added to the 5' of the LHS probe
LHS_suff = '' ## Will be added to the 3' of the LHS probe
RHS_pref = '/5Phos/' ## Will be added to the 5' of the RHS probe
RHS_suff = 'CCCATATAAGAAA' ## Will be added to the 3' of the RHS probe

## Leave as empty strings if you don't want to use them


### 2. Generate probes for each gene

The helper below runs the full probe-design pipeline for one Gene ID, writes all intermediate outputs into that gene's directory, and returns the final selected probes so the notebook can also build a combined summary.

In [7]:
def process_gene(gene_id):
    out_dir = os.path.join(output_root, f'probeDesign_{gene_id}_{mode}')
    os.makedirs(out_dir, exist_ok=True)
    print(f'Processing {gene_id} -> {out_dir}')

    summary = {'gene_ID': gene_id, 'output_dir': out_dir}

    ## Extract regions corresponding to gene of interest (symbol: gene_name, Ensembl ID: gene_ID), subset to feature of interest and convert to bed style dataframe:
    roi_bed = get_region_of_interest(gene_anno, gene_id, gene_id_type='gene_name', feature=mode)
    kmers = generate_kmers(roi_bed, k=probe_length)
    summary['kmers_all'] = kmers.shape[0]

    ## Export unfiltered
    kmers.to_csv(os.path.join(out_dir, 'kmers_all.csv'))

    ## For example, we could have removed all kmers overlapping a repeat/low complexity region within 5 nts of the ligation junction:
    ## In this case we have a lot of possible kmers, so we will remove those with overlaps in any part of the probe:
    kmers = remove_overlaps(kmers, repeats)

    ## Doing the same for gaps in the assembly (very unlikely since we are starting with annotated exons)
    kmers = remove_overlaps(kmers, gaps)

    ## And more importantly, against common polymorphism (SNPs, short indels)
    kmers = remove_overlaps(kmers, snp_db)
    summary['kmers_after_overlap_filters'] = kmers.shape[0]

    ## Get DNA for the transcript
    kmers_bed = pybedtools.BedTool.from_dataframe(kmers)
    kmers_seq = kmers_bed.sequence(fi=fasta, s=True)

    ## We can read in the sequences and simultaneously monitor GC content and count the longest homopolymer stretch
    kmers_seq_stats = get_sequence_stats(kmers_seq.seqfn, probe_length, split_nt)

    ## Combining with our dataframe
    kmers = pd.merge(kmers, kmers_seq_stats, left_index=True, right_index=True)

    ## Check for required nucleotides in specific positions:
    if required_nts is not None:
        kmers['has_required_nts'] = check_for_required_nts(kmers, required_nts)
        print(f'{gene_id} required nts counts:')
        print(kmers['has_required_nts'].value_counts())
        ## Filter for required nucleotides
        kmers = kmers[kmers['has_required_nts'] == True].reset_index(drop=True)
    summary['kmers_after_required_nts'] = kmers.shape[0]

    ## Export kmers before filtering
    kmers.to_csv(os.path.join(out_dir, 'kmers_candidates_unfiltered.csv'))

    if kmers.empty:
        print(f'No candidate kmers remained for {gene_id} after required-nt filtering.')
        selected_probes_df = kmers.copy()
        selected_probes_df['gene_ID'] = gene_id
        selected_probes_df.to_csv(os.path.join(out_dir, 'kmers_selected_probes.csv'))
        summary['kmers_after_GC'] = 0
        summary['selected_probes'] = 0
        return selected_probes_df, summary

    ## Filter for GC content
    kmers = filter_by_GC_content(kmers, min_GC, max_GC)
    summary['kmers_after_GC'] = kmers.shape[0]

    ## Candidate kmers
    kmers.to_csv(os.path.join(out_dir, 'kmers_candidates_filtered.csv'))

    if kmers.empty:
        print(f'No candidate kmers remained for {gene_id} after GC filtering.')
        selected_probes_df = kmers.copy()
        selected_probes_df['gene_ID'] = gene_id
        selected_probes_df.to_csv(os.path.join(out_dir, 'kmers_selected_probes.csv'))
        summary['selected_probes'] = 0
        return selected_probes_df, summary

    ## The first thing to do is to export our sequences in fasta format, so that we can use them for BLAST
    write_fasta(kmers['name'], kmers['transcript_seq'], os.path.join(out_dir, 'kmers_candidates_filtered_transcript_seqs.fa'))
    ## If our probes are meant to be split, we should additionally blast them separately
    ## Note that the LHS/RHS in the transcript are reversed compared to the probe (i.e., the LHS of the transcript is complementary to the RHS of the probe)
    if split_nt is not None:
        ## Make split probes
        kmers['transcript_seq_LHS'] = [seq[0:split_nt] for seq in kmers['transcript_seq']]
        kmers['transcript_seq_RHS'] = [seq[split_nt: probe_length] for seq in kmers['transcript_seq']]

        ## We are exporting the transcript sequence as that's the one that has to be blasted against the human transcriptome
        write_fasta(kmers['name'], kmers['transcript_seq_LHS'], os.path.join(out_dir, 'kmers_candidates_filtered_transcript_seqs_LHS.fa'))
        write_fasta(kmers['name'], kmers['transcript_seq_RHS'], os.path.join(out_dir, 'kmers_candidates_filtered_transcript_seqs_RHS.fa'))

    blast_res = {}
    ## First, blast the full probe
    blast_res['full'] = run_blast(
        fasta=os.path.join(out_dir, 'kmers_candidates_filtered_transcript_seqs.fa'),
        blastdb=blast_db,
        path2blastn=(blast_exec_path + 'blastn'),
        outfile=os.path.join(out_dir, 'kmers_candidates_filtered_blast_output.txt'),
    )

    ## Additionally, if probe is split, blast each side separately
    if split_nt is not None:
        blast_res['LHS'] = run_blast(
            fasta=os.path.join(out_dir, 'kmers_candidates_filtered_transcript_seqs_LHS.fa'),
            blastdb=blast_db,
            path2blastn=(blast_exec_path + 'blastn'),
            outfile=os.path.join(out_dir, 'kmers_candidates_filtered_blast_output_LHS.txt'),
        )
        blast_res['RHS'] = run_blast(
            fasta=os.path.join(out_dir, 'kmers_candidates_filtered_transcript_seqs_RHS.fa'),
            blastdb=blast_db,
            path2blastn=(blast_exec_path + 'blastn'),
            outfile=os.path.join(out_dir, 'kmers_candidates_filtered_blast_output_RHS.txt'),
        )

    for k in blast_res.keys():
        print(('The following genes were detected in mode: ' + k))
        print(blast_res[k]['sgeneid'].value_counts().head(10))

    offtargets = []
    for k in blast_res.keys():
        offtargets += detect_offtargets(blast_res[k], gene_id, min_mismatches=min_mismatches)
    ## Remove redundancies
    offtargets = list(set(offtargets))
    summary['offtargets'] = len(offtargets)

    ## Remove off-targets
    kmers = kmers[kmers['name'].isin(offtargets) == False].reset_index(drop=True)
    summary['kmers_after_offtarget_filter'] = kmers.shape[0]

    if kmers.empty:
        print(f'No candidate kmers remained for {gene_id} after off-target filtering.')
        selected_probes_df = kmers.copy()
        selected_probes_df['gene_ID'] = gene_id
        selected_probes_df.to_csv(os.path.join(out_dir, 'kmers_selected_probes.csv'))
        summary['selected_probes'] = 0
        return selected_probes_df, summary

    ## Sort in increasing homopolymer length
    kmers = kmers.sort_values('longest_homopolymer', ascending=True).reset_index(drop=True)

    ## Select final probes, using a larger spacing to spread them across the gene
    selected_probes_list = []
    df = kmers.copy()
    while len(selected_probes_list) < n_desired_probes and not df.empty:
        selected_probes_list.append(df.iloc[0:1, :])
        if df.shape[0] == 1:
            break
        df = remove_overlapping_probes(
            df,
            df['name'].iloc[0],
            offset=probe_selection_offset,
        ).reset_index(drop=True).copy()

    if selected_probes_list:
        selected_probes_df = pd.concat(selected_probes_list, axis=0).reset_index(drop=True)
    else:
        selected_probes_df = pd.DataFrame(columns=kmers.columns)

    if split_nt is not None and not selected_probes_df.empty:
        ## Make split probes (and add adapters if provided)
        selected_probes_df['probe_seq_LHS'] = [
            (LHS_pref + seq[0:split_nt] + LHS_suff)
            for seq in selected_probes_df['probe_seq']
        ]
        selected_probes_df['probe_seq_RHS'] = [
            (RHS_pref + seq[split_nt: probe_length] + RHS_suff)
            for seq in selected_probes_df['probe_seq']
        ]

    ## Also add gene_ID for completeness
    selected_probes_df['gene_ID'] = gene_id

    ## Export selected probes as dataframe:
    selected_probes_df.to_csv(os.path.join(out_dir, 'kmers_selected_probes.csv'))
    summary['selected_probes'] = selected_probes_df.shape[0]
    return selected_probes_df, summary


probe_results = {}
summary_rows = []
for gene_id in gene_ids:
    selected_probes_df, summary = process_gene(gene_id)
    probe_results[gene_id] = selected_probes_df
    summary_rows.append(summary)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(output_root, f'probeDesign_{mode}_summary.csv'), index=False)

all_selected_probes_df = pd.concat(probe_results.values(), ignore_index=True) if probe_results else pd.DataFrame()
all_selected_probes_df.to_csv(os.path.join(output_root, f'probeDesign_{mode}_selected_probes_all_genes.csv'), index=False)

summary_df


Processing CDKN2B-AS1 -> ../sample_run/probeDesign_CDKN2B-AS1_transcript
CDKN2B-AS1 required nts counts:
has_required_nts
False    461295
True     196158
Name: count, dtype: int64
The following genes were detected in mode: full
sgeneid
CDKN2B-AS1      151854
UBA52             9562
LINC03007         5600
UBB               4200
UBC               2366
LOC105370003      1372
PRELID2           1176
UST               1176
CFAP20DC          1176
NRXN3              918
Name: count, dtype: int64
The following genes were detected in mode: LHS
sgeneid
CDKN2B-AS1    151462
UBA52           6678
LINC03007       2128
CYRIB           1176
UBB             1092
PTPRD            770
MELK             742
UBA52P6          728
UROS             616
KALRN            612
Name: count, dtype: int64
The following genes were detected in mode: RHS
sgeneid
CDKN2B-AS1      150874
UBA52             5460
LINC03007         2800
UBB               1512
PTPRD              770
UBA52P6            742
UST                686
C

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'Series'